# Semaine 6 : optimisation sous contraintes (scipy.optimize et Pyomo)

Objectifs du cours (10 novembre) :
1. introduction à l'optimisation non linéaire **sous contraintes** avec `scipy.optimize` et **Pyomo** ;
2. application à la **régression** (OLS et LAD écrites comme des problèmes d'optimisation) ;
3. application à l'**optimisation du prix**.

Plan :
- partie A : régression OLS / LAD avec Pyomo, avec et sans contraintes (script `w6_regression.py`) ;
- partie B : prix optimal sous contrainte de capacité (80 sièges), avec scipy (SLSQP), Pyomo et la méthode de Lagrange ;
- partie C : simulation (script `w6_simulation.py`).

Installation (une fois, dans le venv) : `pip install pyomo highspy` (HiGHS résout les problèmes linéaires et quadratiques ; ipopt n'est pas nécessaire ici).

In [ ]:
import os, sys, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pyomo.environ as pyo
from scipy.optimize import minimize
os.chdir('..')                      # racine du dépôt
sys.path.insert(0, '.')
import duopoly as d
plt.style.use('ggplot')

# A. La régression comme problème d'optimisation (Pyomo)

- **OLS** : minimiser la somme des erreurs au carré : problème quadratique convexe (une seule solution).
- **LAD** : minimiser la somme des erreurs absolues ; astuce du cours : `e = e⁺ − e⁻` avec `e⁺, e⁻ ≥ 0`, ce qui donne un problème **linéaire**.
- **Contraintes métier** possibles : `b ≤ −0,005` (prix plus haut = moins de demande) et `c ≥ 0` (concurrent plus cher = plus de demande).

Données : les 16 compétitions d'entraînement (avant 16/10, 21/10, 24/10), saisons 1-50 pour estimer, 51-100 pour tester, 5 blocs de 20 jours.

In [ ]:
# environ 2 minutes (16 000 lignes par bloc, 4 modèles par bloc)
%run W6-Notebooks/w6_regression.py

**Constats** (`w6_regression_results.csv`) :
- Pyomo + HiGHS redonne **exactement** les coefficients de statsmodels (a = 0,635, b = −0,0115, c = 0,0059 pour les jours 1-20) : c'est bien le même problème.
- Les contraintes `b ≤ −0,005, c ≥ 0` **ne changent rien** sur nos données communes : elles sont déjà respectées. Elles seraient utiles pour un modèle estimé sur peu de données (pendant la saison), où b peut sortir positif.
- **LAD ne marche pas ici** : 60 % des jours ont une demande de 0, donc la « médiane » est 0 et LAD trouve a = b = c = 0. Avec contraintes, b reste collé à −0,005 et le prix optimal tombe à 15-28 €. Erreur absolue plus petite, mais erreur au carré bien plus grande : LAD prédit « 0 vente », ce qui est inutile pour fixer un prix. On garde OLS.

# B. Prix optimal sous contrainte de capacité

Jusqu'ici, chaque jour on prenait le prix qui maximise le revenu du jour : `p* = −(a + c·pc) / (2b)`, sans regarder les sièges restants.

Nouveau problème, pour les jours restants s = t … 100 :

$$\max_{p_t..p_{100}} \sum_s p_s\, d_s(p_s) \quad\text{avec}\quad d_s = a_s + b_s p_s + c_s\, \hat{pc}$$
$$\text{s.c.}\quad \sum_s d_s \le \text{sièges restants},\qquad 15 \le p_s \le 150,\qquad d_s \ge 0$$

(`pc` = prix concurrent prédit, gardé constant). On ne joue que le prix du jour t, et on refait le calcul au prochain changement de prix.

**Méthode de Lagrange** : on donne une « valeur » λ ≥ 0 à chaque siège ; le problème devient `max Σ (p_s − λ)·d_s`, d'où `p_s = p*_s + λ/2`.
Si les sièges suffisent, λ = 0 (on retrouve p*). Sinon, on cherche λ (dichotomie) pour que les ventes attendues = sièges restants : **tous les prix montent de λ/2**.

In [ ]:
def plan_slsqp(t, pc, remaining):
    days = np.arange(t, 101)
    abc = np.array([d.get_ols_params(s) for s in days])
    a, b = abc[:, 0] + abc[:, 2] * pc, abc[:, 1]
    x0 = np.clip(-a / (2 * b), 15, 150)
    res = minimize(lambda x: -np.sum(x * (a + b * x)), x0, method='SLSQP', bounds=[(15, 150)] * len(days),
                   constraints=[{'type': 'ineq', 'fun': lambda x: remaining - np.sum(a + b * x)},
                                {'type': 'ineq', 'fun': lambda x: a + b * x}])
    return days, res.x, a + b * res.x

def plan_pyomo(t, pc, remaining):
    days = list(range(t, 101))
    abc = {s: d.get_ols_params(s) for s in days}
    m = pyo.ConcreteModel()
    m.p = pyo.Var(days, bounds=(15, 150))
    dem = lambda s: abc[s][0] + abc[s][2] * pc + abc[s][1] * m.p[s]
    m.cap = pyo.Constraint(expr=sum(dem(s) for s in days) <= remaining)
    m.pos = pyo.Constraint(days, rule=lambda m, s: dem(s) >= 0)
    # objectif écrit en polynôme (a·p + b·p²) : HiGHS accepte alors ce problème quadratique
    m.obj = pyo.Objective(expr=sum((abc[s][0] + abc[s][2] * pc) * m.p[s] + abc[s][1] * m.p[s] ** 2 for s in days),
                          sense=pyo.maximize)
    pyo.SolverFactory('highs').solve(m)
    return np.array([m.p[s].value for s in days])

t, pc = 30, 50
d.CAPACITY_OPT_METHOD = 'lagrange'
for R in (80, 40, 20, 10):
    days, x, dem = plan_slsqp(t, pc, R)
    print(f'sièges restants {R:2d} : prix du jour  SLSQP {x[0]:6.1f} | Pyomo {plan_pyomo(t, pc, R)[0]:6.1f} | '
          f'Lagrange {d.get_capacity_optimal_price(t, pc, R):6.1f} | sans contrainte {d.get_ols_optimal_price(t, pc, 0):6.1f} | ventes attendues {dem.sum():5.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for R in (80, 30, 15, 8):
    days, x, _ = plan_slsqp(30, 50, R)
    ax.plot(days, x, label=f'{R} sièges restants')
ax.set_xlabel('jour'); ax.set_ylabel('prix prévu (€)'); ax.set_title('Plan de prix au jour 30 (concurrent à 50 €)')
ax.legend(); plt.show()

In [ ]:
# temps de calcul (contrainte DPC : < 0,2 s en moyenne par appel)
for meth in ('lagrange', 'slsqp'):
    d.CAPACITY_OPT_METHOD = meth
    t0 = time.perf_counter()
    for k in range(200):
        d.get_capacity_optimal_price(2 + k % 90, 50.0, 20)
    print(meth, f'{(time.perf_counter() - t0) / 200 * 1e3:.2f} ms par calcul')
d.CAPACITY_OPT_METHOD = 'lagrange'

Les trois méthodes donnent le même prix, sauf dans les cas extrêmes où même 150 € ne suffit pas : SLSQP échoue et Lagrange renvoie 150 €.
Lagrange est environ 20 fois plus rapide (0,3 ms contre 5,6 ms, max 157 ms) : c'est elle qui est utilisée dans `duopoly.py`.
Pyomo n'est pas utilisable sur DPC (pas installé, pas de solveur) : il sert à vérifier hors ligne.

# C. Simulation (84 compétitions : 46 de Simon, 38 d'Alois, saisons 51-99)

`duopoly.py` a 2 nouvelles options : `USE_CAPACITY_OPT` (prix optimisé sous contrainte) et `FOLLOW_BAND` (garder la correction par les marges de la courbe de remplissage).

In [ ]:
# environ 4 minutes
%run W6-Notebooks/w6_simulation.py

**Résultats** (`w6_simulation_summary.csv`, revenu moyen par saison) :

| Version | Tout | Jamais vu | Sièges | Saisons gagnées / perdues vs W4 |
|---|---|---|---|---|
| W4 actuelle | 2045 | 2024 | 52,4 | – |
| **W6 prix optimisé + marges** | **2055** | **2030** | 52,4 | **11,6 % / 2,5 %** |
| W6 prix optimisé seul | 1888 | 1824 | 32,1 | 26,6 % / 68,9 % |
| W6 optimisé + marge basse seule | 2052 | 2029 | 52,5 | 12,0 % / 7,9 % |

- Avec les marges, le prix optimisé apporte un petit gain (+0,5 %) et presque jamais de perte : il remonte les prix quand les sièges deviennent rares (surtout dans les runs à forte demande : 21/10 de Simon +1,9 %).
- **Seul**, il est très bon quand la demande est forte (21/10 d'Alois 3909 contre 3755, W2 B +19 %) mais très mauvais quand elle est faible (24/10 d'Alois 403 contre 871, 31/10 227 contre 471) : le modèle commun ne sait pas que la demande de cette compétition est faible, et rien ne fait baisser le prix. Les marges, elles, le font.
- Leçon : la contrainte de capacité ne sert que si le modèle connaît le **niveau de demande** de la compétition en cours. C'est le sujet des prochains cours (apprentissage pendant la saison le 17/11, regrouper les compétitions par niveau de demande le 5/01).